# Sesi 10 · Menyiapkan Data untuk Dashboard

**Kursus:** Data Analysis dengan BigQuery, Python & BI

> **Opsional.** Di kelas, 3 file CSV dibagikan instruktur lewat Google Drive. Notebook ini menunjukkan cara membuatnya sendiri dari BigQuery (dipakai instruktur untuk menyiapkan file, dan bisa dicoba peserta jalur penuh).

Notebook ini mengekspor data TheLook menjadi **3 file CSV** yang akan dihubungkan di Power BI:

| File | Isi | Satu baris = |
| --- | --- | --- |
| `penjualan.csv` | Tabel fakta: tanggal, harga jual, biaya | satu barang terjual |
| `produk.csv` | Kategori, brand, department | satu produk |
| `pelanggan.csv` | Umur, kelompok umur, gender, negara, traffic source | satu pelanggan |

Kenapa tiga file dan bukan satu? Supaya kita bisa berlatih membuat **relasi** antar tabel di tools dashboard, persis seperti JOIN di sesi 3.

Jalankan semua sel (**Runtime → Run all**), lalu simpan ketiga file yang terunduh ke satu folder di laptop, misalnya `Documents/thelook-dashboard/`.

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import pandas as pd
from google.cloud import bigquery
from google.colab import files

PROJECT_ID = "isi-project-id-kamu"   # <-- GANTI dengan Project ID-mu
client = bigquery.Client(project=PROJECT_ID)

def jalankan_query(sql):
    """Menjalankan query SQL di BigQuery dan mengembalikan hasilnya sebagai DataFrame."""
    return client.query(sql).to_dataframe()

## 1. Tabel fakta: penjualan

Hanya barang yang **tidak** `Cancelled`/`Returned`, dan tanpa bulan berjalan. Jadi semua angka di dashboard adalah penjualan yang sah.

In [ ]:
penjualan = jalankan_query("""
SELECT
  oi.id AS order_item_id,
  oi.order_id,
  oi.user_id,
  oi.product_id,
  DATE(oi.created_at) AS order_date,
  oi.sale_price,
  p.cost
FROM `bigquery-public-data.thelook_ecommerce.order_items` AS oi
JOIN `bigquery-public-data.thelook_ecommerce.products` AS p ON oi.product_id = p.id
WHERE oi.status NOT IN ('Cancelled', 'Returned')
  AND DATE(oi.created_at) < DATE_TRUNC(CURRENT_DATE(), MONTH)
""")
print(penjualan.shape)
penjualan.head()

## 2. Tabel dimensi: produk dan pelanggan

Hanya produk dan pelanggan yang muncul di tabel penjualan, supaya file tetap kecil.

In [ ]:
produk = jalankan_query("""
SELECT
  id AS product_id,
  name AS product_name,
  category,
  COALESCE(brand, 'Tidak diketahui') AS brand,
  department,
  retail_price
FROM `bigquery-public-data.thelook_ecommerce.products`
""")
produk = produk[produk["product_id"].isin(penjualan["product_id"])]
print(produk.shape)
produk.head()

In [ ]:
pelanggan = jalankan_query("""
SELECT
  id AS user_id,
  age,
  CASE
    WHEN age < 18 THEN '1. Di bawah 18'
    WHEN age < 25 THEN '2. 18-24'
    WHEN age < 35 THEN '3. 25-34'
    WHEN age < 45 THEN '4. 35-44'
    WHEN age < 55 THEN '5. 45-54'
    ELSE '6. 55+'
  END AS age_band,
  gender,
  country,
  traffic_source
FROM `bigquery-public-data.thelook_ecommerce.users`
""")
pelanggan = pelanggan[pelanggan["user_id"].isin(penjualan["user_id"])]
print(pelanggan.shape)
pelanggan.head()

## 3. Cek cepat sebelum diekspor

Angka-angka ini akan kita cocokkan dengan kartu KPI di dashboard. Kalau angka di dashboard berbeda, ada yang salah dengan relasi atau rumusnya.

In [ ]:
pendapatan = penjualan["sale_price"].sum()
jumlah_pesanan = penjualan["order_id"].nunique()
print(f"Pendapatan     : {pendapatan:,.0f} USD")
print(f"Jumlah pesanan : {jumlah_pesanan:,}")
print(f"AOV            : {pendapatan / jumlah_pesanan:,.2f} USD")
print(f"Margin         : {(penjualan['sale_price'] - penjualan['cost']).sum():,.0f} USD")
print(f"Periode        : {penjualan['order_date'].min()} s.d. {penjualan['order_date'].max()}")

## 4. Ekspor ke CSV dan unduh

Browser mungkin bertanya apakah boleh mengunduh beberapa file sekaligus. Pilih **Izinkan/Allow**.

In [ ]:
penjualan.to_csv("penjualan.csv", index=False)
produk.to_csv("produk.csv", index=False)
pelanggan.to_csv("pelanggan.csv", index=False)

for nama_file in ["penjualan.csv", "produk.csv", "pelanggan.csv"]:
    files.download(nama_file)